# 02. Layer 1 và Layer 2

Embed BGE-M3 (nên bật GPU), tính s và p, grid search λ×τ trên dev theo từng tenant, báo cáo trên test.

In [1]:
import os, sys, subprocess
from importlib import metadata
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", "https://github.com/vinhlee9x/conflict-detection-obsolescence-forecasting.git", "/content/repo"], check=True)
    os.chdir("/content/repo")
    subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
    core = ("numpy", "pandas", "scipy", "scikit-learn", "pyarrow")
    def installed_versions():
        found = {}
        for name in core:
            try:
                found[name] = metadata.version(name)
            except metadata.PackageNotFoundError:
                found[name] = None
        return found
    before = installed_versions()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    if installed_versions() != before:
        print("Đã cài phiên bản mới của numpy/pandas/scipy. Runtime tự khởi động lại, đợi vài giây rồi chạy lại notebook từ đầu.")
        os.kill(os.getpid(), 9)
else:
    while not os.path.exists("src") and os.getcwd() != "/":
        os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Thư mục làm việc:", os.getcwd())

Thư mục làm việc: /content/repo


In [2]:
# Tuỳ chọn: giữ data/ và artifacts/ trên Google Drive giữa các session Colab
USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    target = "/content/drive/MyDrive/conflict-detection"
    for folder in ("data/processed", "artifacts"):
        os.makedirs(f"{target}/{folder}", exist_ok=True)
        subprocess.run(f"cp -rn {folder}/. {target}/{folder}/ && rm -rf {folder} && ln -s {target}/{folder} {folder}", shell=True, check=True)

Mounted at /content/drive


In [3]:
from src import pipeline
import pandas as pd
summary = pipeline.embed("kb", backend="bge-m3")
summary

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (9 outputs left out)

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/61 [00:00<?, ?it/s]

{'backend': 'bge-m3',
 'documents': 972,
 'chunks': 972,
 'seconds': 27.728477417000022}

## ViNLI: cosine có tách được mâu thuẫn không?

In [4]:
pipeline.vinli(backend="bge-m3", per_label=1000)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/283 [00:00<?, ?it/s]

{'backend': 'bge-m3',
 'split': 'test',
 'n': 2264,
 'seconds': 32.392279357999996,
 'mean_cosine': {'contradiction': 0.6860887795646284,
  'entailment': 0.7741017802158991,
  'neutral': 0.6889188364819363},
 'auc_entailment_vs_contradiction': 0.7233649932157394,
 'auc_contradiction_vs_neutral': 0.4956264657757195}

## Calibration và ablation

In [5]:
results = pipeline.layer2("kb")
pd.DataFrame(results["test"])[["scope", "method_label", "lambda", "tau", "precision", "recall", "f1", "f1_ci95", "pr_auc"]]

,scope,method_label,lambda,tau,precision,recall,f1,f1_ci95,pr_auc
0,all,Chỉ s (λ = 1),NaN,NaN,0.546512,0.824561,0.657343,"[0.573527, 0.744195]",0.636459
1,all,Chỉ p (λ = 0),NaN,NaN,0.480000,0.421053,0.448598,"[0.326531, 0.56672]",0.417172
2,all,"Fusion, λ và τ theo tenant",NaN,NaN,0.546512,0.824561,0.657343,"[0.562938, 0.744834]",0.636459
3,all,"Fusion, λ và τ chung",NaN,NaN,0.529412,0.789474,0.633803,"[0.532371, 0.725928]",0.633509
4,policy,Chỉ s (λ = 1),1.0,0.77,0.534884,0.741935,0.621622,"[0.474576, 0.752949]",0.650115
5,policy,Chỉ p (λ = 0),0.0,0.30,0.461538,0.387097,0.421053,"[0.255319, 0.571429]",0.426818
6,policy,"Fusion, λ và τ theo tenant",1.0,0.77,0.534884,0.741935,0.621622,"[0.484824, 0.734316]",0.650115
7,policy,"Fusion, λ và τ chung",1.0,0.75,0.511111,0.741935,0.605263,"[0.457719, 0.725]",0.650115
8,tech,Chỉ s (λ = 1),1.0,0.68,0.558140,0.923077,0.695652,"[0.561404, 0.8]",0.630247
9,tech,Chỉ p (λ = 0),0.0,0.30,0.500000,0.461538,0.480000,"[0.294082, 0.645209]",0.424843


In [6]:
pd.DataFrame(results["test_by_variant"])

,variant_type,label,n,flagged_s_only,flagged_p_only,flagged_fusion,flagged_global,mean_s,mean_p
0,conflict,1,30,1.00000,0.800000,1.00000,1.000000,0.968104,0.347499
1,hidden_conflict,1,27,0.62963,0.000000,0.62963,0.555556,0.883404,0.000000
2,lexical_overlap,0,26,0.00000,0.000000,0.00000,0.076923,0.783685,0.000018
3,paraphrase,0,35,1.00000,0.742857,1.00000,1.000000,0.955444,0.332276
4,random,0,85,0.00000,0.000000,0.00000,0.000000,0.665641,0.001055
5,same_template,0,25,0.16000,0.000000,0.16000,0.120000,0.836760,0.023007


In [7]:
{tenant: {k: cfg[k] for k in ("FUSION_LAMBDA", "CONFLICT_THRESHOLD_TAU", "dev_f1")} for tenant, cfg in results["config"]["tenants"].items()}

{'policy': {'FUSION_LAMBDA': 1.0,
  'CONFLICT_THRESHOLD_TAU': 0.77,
  'dev_f1': 0.708029197080292},
 'tech': {'FUSION_LAMBDA': 1.0,
  'CONFLICT_THRESHOLD_TAU': 0.68,
  'dev_f1': 0.6645367412140575}}

## Luồng end-to-end: tài liệu mới → candidate → xác nhận

Lấy 3 tài liệu ban hành gần nhất làm ví dụ; người dùng xác nhận candidate đầu tiên.

In [8]:
from src.conflict import confirm_conflicts, detect_candidates
from src.graph import KnowledgeGraph
from src.io_utils import read_parquet
from src.paths import DATA_PROCESSED
docs = read_parquet(DATA_PROCESSED / "kb_docs.parquet")
edges = read_parquet(DATA_PROCESSED / "kb_edges.parquet")
new_ids = docs.sort_values("issued_at").doc_id.tail(3).tolist()
graph = KnowledgeGraph(docs.doc_id.tolist(), edges)
candidates = detect_candidates(new_ids, docs, pipeline.load_vectors("kb"), graph, results["config"], top_k=15)
candidates

,doc_id,candidate_id,s,p,score
0,POL-003-v2,POL-003-v1,0.980346,0.355525,0.979126
1,SOP-104-v2,SOP-104-v1,0.963500,0.356590,0.944263
2,POL-314-v1,POL-307-v1,0.953822,0.002032,0.922860
3,POL-314-v1,POL-056-v1,0.935608,0.001317,0.884221
4,POL-314-v1,POL-056-v2,0.928042,0.001060,0.868170
5,POL-003-v2,POL-204-v1,0.900017,0.000000,0.808720
6,SOP-104-v2,SOP-079-v1,0.900028,0.006697,0.789895
7,SOP-104-v2,SOP-055-v2,0.895231,0.000901,0.778228
8,SOP-104-v2,SOP-093-v1,0.893151,0.000378,0.773170
9,SOP-104-v2,SOP-244-v1,0.891782,0.013766,0.769840


In [9]:
confirm_conflicts(graph, candidates.head(1))
graph.edges[graph.edges.type == "CONFLICTS_WITH"]

,src,dst,type
3891,POL-003-v2,POL-003-v1,CONFLICTS_WITH


In [11]:
import os
if os.path.exists("data/processed/jira_pairs.parquet"):
    jira = pipeline.layer2("jira")
    display(pd.DataFrame(jira["test"])[["scope", "method_label", "precision", "recall", "f1", "pr_auc"]])